In [1]:
%load_ext autoreload
%autoreload 2
import time
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score, roc_auc_score
from src.config import *
from src.experiments import CV, build_models, build_ensembles

full = pd.read_csv(BINARY_FEATURES_TRAIN_FULL_CSV)
subset_files = set(pd.read_csv(BINARY_FEATURES_TRAIN_CSV)["filename"])
feature_cols = [c for c in full.columns if c not in ("filename", "label")]

X, y = full[feature_cols].values, full["label"].values
in_subset = full["filename"].isin(subset_files).values
subtype = full["filename"].str.replace(r"\d+\.jpg$", "", regex=True).values
print(X.shape, np.bincount(y), "| subset images:", in_subset.sum())

(15195, 230) [7725 7470] | subset images: 5170


In [2]:
BASE = ["HistGradBoost", "SVM (RBF)", "XGBoost", "CatBoost"]       # same members as in exp5

exp5 = pd.read_csv(RESULTS_DIR / "exp5_binary_ensembles.csv")
params = {r.model: eval(r.params, {"np": np}) for r in exp5.itertuples() if r.model in BASE}

all_models = build_models()
base = {name: clone(all_models[name]).set_params(**params[name]) for name in BASE}
MODELS = {
    "Stacking (LogReg)": build_ensembles(base)["Stacking (LogReg)"],
    "SVM (RBF)": base["SVM (RBF)"],
}
params

{'HistGradBoost': {},
 'SVM (RBF)': {'clf__C': np.float64(11.290133559092672),
  'clf__gamma': np.float64(0.0020914981329035616)},
 'XGBoost': {'clf__colsample_bytree': np.float64(0.685222133909267),
  'clf__learning_rate': np.float64(0.08733914216812769),
  'clf__max_depth': 6,
  'clf__min_child_weight': 5,
  'clf__n_estimators': 430,
  'clf__subsample': np.float64(0.7641531692142519)},
 'CatBoost': {'clf__depth': 7,
  'clf__iterations': 587,
  'clf__l2_leaf_reg': np.float64(2.0497980520950176),
  'clf__learning_rate': np.float64(0.08046552389871331)}}

In [3]:
def run_fold(fold, model_name, data, tr, te):
    """Train on the subset part (or all) of the training fold; test on the same full test fold."""
    if data == "subset":
        tr = tr[in_subset[tr]]
    model = clone(MODELS[model_name])
    start = time.perf_counter()
    model.fit(X[tr], y[tr])
    fit_min = (time.perf_counter() - start) / 60

    pred, score = model.predict(X[te]), model.decision_function(X[te])
    mel = subtype[te] == "mel"
    return {"model": model_name, "train data": data, "fold": fold, "n_train": len(tr),
            "balanced_accuracy": balanced_accuracy_score(y[te], pred),
            "accuracy": accuracy_score(y[te], pred),
            "sensitivity": recall_score(y[te], pred, pos_label=1),
            "specificity": recall_score(y[te], pred, pos_label=0),
            "auc": roc_auc_score(y[te], score),
            "recall_mel": (pred[mel] == 1).mean(),
            "fit_min": fit_min}

In [4]:
jobs = [(fold, name, data, tr, te)
        for fold, (tr, te) in enumerate(CV.split(X, y))
        for name in MODELS                     # stacking first: the longest jobs start early
        for data in ["full", "subset"]]

rows = Parallel(n_jobs=-2, verbose=10)(delayed(run_fold)(*job) for job in jobs)
folds = pd.DataFrame(rows)
folds.to_csv(RESULTS_DIR / "exp6_full_train_folds.csv", index=False)

[Parallel(n_jobs=-2)]: Using backend LokyBackend with 11 concurrent workers.
[Parallel(n_jobs=-2)]: Done   2 out of  20 | elapsed:   16.6s remaining:  2.5min
[Parallel(n_jobs=-2)]: Done   5 out of  20 | elapsed:  1.3min remaining:  4.0min
[Parallel(n_jobs=-2)]: Done   8 out of  20 | elapsed:  2.3min remaining:  3.4min
[Parallel(n_jobs=-2)]: Done  11 out of  20 | elapsed: 21.1min remaining: 17.2min
[Parallel(n_jobs=-2)]: Done  14 out of  20 | elapsed: 21.4min remaining:  9.2min
[Parallel(n_jobs=-2)]: Done  17 out of  20 | elapsed: 25.3min remaining:  4.5min
[Parallel(n_jobs=-2)]: Done  20 out of  20 | elapsed: 25.7min finished


In [5]:
metrics = ["balanced_accuracy", "accuracy", "sensitivity", "specificity", "auc", "recall_mel", "fit_min", "n_train"]
summary = folds.groupby(["model", "train data"])[metrics].mean().round(3)
display(summary)

# Paired gain per fold (same test images), full - subset
paired = folds.pivot_table(index=["model", "fold"], columns="train data",
                           values=["balanced_accuracy", "auc", "recall_mel"])
gain = (paired.xs("full", axis=1, level=1) - paired.xs("subset", axis=1, level=1))
gain.groupby("model").agg(["mean", "std"]).round(3)

balanced_accuracy  accuracy  sensitivity  \
model             train data                                             
SVM (RBF)         full                    0.830     0.830        0.826   
                  subset                  0.810     0.810        0.793   
Stacking (LogReg) full                    0.837     0.837        0.831   
                  subset                  0.819     0.819        0.802   

                              specificity    auc  recall_mel  fit_min  n_train  
model             train data                                                    
SVM (RBF)         full              0.835  0.908       0.706    0.769  12156.0  
                  subset            0.827  0.887       0.641    0.070   4136.0  
Stacking (LogReg) full              0.843  0.915       0.712   24.923  12156.0  
                  subset            0.836  0.893       0.647   20.863   4136.0

auc        balanced_accuracy        recall_mel       
                    mean    std              mean    std       mean    std
model                                                                     
SVM (RBF)          0.021  0.003             0.020  0.003      0.064  0.025
Stacking (LogReg)  0.022  0.002             0.018  0.004      0.065  0.016